# 06 - Branch and Bound

**Goal:** Understand how branch-and-bound combines LP relaxations, branching, and bounds to certify an optimal integer solution.

The previous notebook introduced the integer hull and showed that optimizing over it is equivalent to solving the integer program. Explicitly constructing that hull is usually impractical. **Branch-and-bound** takes a different route: it recursively divides the LP relaxation into smaller subproblems and discards any subproblem that cannot improve the best integer solution found so far.

This notebook will:

1. Reuse the two-dimensional example from the integer-hull notebook
2. Derive the branching disjunction and the three pruning rules
3. Follow a complete search tree by hand
4. Run a small reusable implementation and inspect its node trace

We consider maximization problems of the form

$$
\begin{aligned}
\max \quad & c^\top x \\
\text{subject to} \quad & Ax\le b,\\
& x_i\in\mathbb{Z} && \text{for } i\in I,
\end{aligned}
$$

where variables outside $I$ may remain continuous. When $I=\{1,\ldots,n\}$, this is a pure integer linear program; otherwise it is a mixed-integer linear program.

## Relaxations provide bounds

At a node of the search tree, let $Q$ denote the part of the original relaxation that remains after adding the node's branching restrictions. Dropping integrality gives the node LP

$$
z_{\mathrm{LP}}(Q)=\max\{c^\top x\mid x\in Q\}.
$$

Because every integer-feasible point in $Q$ is also feasible for this LP,

$$
z_{\mathrm{IP}}(Q)\le z_{\mathrm{LP}}(Q).
$$

Thus, for a **maximization** problem, the node's LP value is an **upper bound** on every integer solution below that node.

During the search, the best integer-feasible solution found so far is called the **incumbent**. Its value $z_{\mathrm{inc}}$ is a global **lower bound** on the optimum. The search is complete when every remaining node is either solved or shown unable to contain a solution better than the incumbent.

## Running example

We continue with the rational polytope from the previous notebook:

$$
P=\left\{x\in\mathbb{R}^2\ \middle|\
\begin{aligned}
x_1&\ge0,\\
x_2&\ge0,\\
2x_1+x_2&\le\frac{17}{2},\\
x_1+2x_2&\le\frac{17}{2}
\end{aligned}
\right\}.
$$

This time, our optimization problem is

$$
\max\{x_1+x_2\mid x\in P\cap\mathbb{Z}^2\}.
$$

The LP relaxation is the **root node** of the branch-and-bound tree.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import linprog

from src import hrep_to_vertices_2d, lattice_points_in_polytope_2d

A = np.array(
    [
        [-1.0, 0.0],       # x1 >= 0
        [0.0, -1.0],       # x2 >= 0
        [2.0, 1.0],        # 2*x1 + x2 <= 17/2
        [1.0, 2.0],        # x1 + 2*x2 <= 17/2
    ]
)
b = np.array([0.0, 0.0, 8.5, 8.5])
c = np.array([1.0, 1.0])
free_bounds = [(None, None)] * 2

root_relaxation = linprog(-c, A_ub=A, b_ub=b, bounds=free_bounds, method="highs")
root_x = root_relaxation.x
root_bound = -root_relaxation.fun

vertices = hrep_to_vertices_2d(A, b)
lattice_points = lattice_points_in_polytope_2d(A, b)
closed_vertices = np.vstack([vertices, vertices[0]])

fig, ax = plt.subplots(figsize=(7, 6))
ax.fill(
    closed_vertices[:, 0],
    closed_vertices[:, 1],
    color="tab:blue",
    alpha=0.15,
    label="$P$ (root LP relaxation)",
)
ax.plot(closed_vertices[:, 0], closed_vertices[:, 1], color="tab:blue")
ax.scatter(
    lattice_points[:, 0],
    lattice_points[:, 1],
    color="black",
    s=25,
    zorder=3,
    label=r"$P\cap\mathbb{Z}^2$",
)
ax.scatter(
    *root_x,
    marker="x",
    s=120,
    linewidths=2,
    color="tab:red",
    zorder=4,
    label="root LP optimum",
)
objective_x = np.linspace(0, 5, 100)
ax.plot(
    objective_x,
    root_bound - objective_x,
    linestyle="--",
    color="tab:red",
    label=rf"$x_1+x_2={root_bound:.2f}$",
)
ax.set_xlim(-0.5, 5.0)
ax.set_ylim(-0.5, 5.0)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.set_title("The root relaxation has a fractional optimum")
ax.grid(True)
ax.legend()
plt.show()

print("Root LP solution:", root_x)
print("Root LP upper bound:", root_bound)

Both slanted constraints are tight at the root optimum, so

$$
x^{\mathrm{LP}}=\left(\frac{17}{6},\frac{17}{6}\right),
\qquad
z_{\mathrm{LP}}=\frac{17}{3}.
$$

The value $17/3$ is a valid upper bound, but the fractional point is not feasible for the integer program. We therefore split the feasible set.

## Branching on a fractional variable

Suppose an integer-constrained component has the fractional LP value $x_j^*$. Every integer point satisfies the disjunction

$$
x_j\le\lfloor x_j^*\rfloor
\qquad\text{or}\qquad
x_j\ge\lceil x_j^*\rceil.
$$

The open interval between these two bounds contains no integer value, so the two child nodes together preserve **every** integer-feasible point of the parent. Their continuous relaxations may be smaller than the parent relaxation, which can improve the LP bounds.

At the root, we choose $x_1^*=17/6$ and create

$$
Q_L=P\cap\{x_1\le2\},
\qquad
Q_R=P\cap\{x_1\ge3\}.
$$

The choice of variable affects efficiency, but not the validity of the split. Our educational implementation simply chooses the first fractional integer-constrained variable.

In [ ]:
def solve_relaxation(variable_bounds):
    # This helper is deliberately specific to the running example.
    result = linprog(-c, A_ub=A, b_ub=b, bounds=variable_bounds, method="highs")
    return result.x, -result.fun


left_x, left_bound = solve_relaxation([(None, 2.0), (None, None)])
right_x, right_bound = solve_relaxation([(3.0, None), (None, None)])

print(f"Left child:  x = {left_x}, upper bound = {left_bound:.2f}")
print(f"Right child: x = {right_x}, upper bound = {right_bound:.2f}")

The split reduces the root bound, but neither child LP solution is integral:

$$
\begin{aligned}
x^L&=\left(2,\frac{13}{4}\right), & z_{\mathrm{LP}}(Q_L)&=\frac{21}{4},\\
x^R&=\left(3,\frac{5}{2}\right), & z_{\mathrm{LP}}(Q_R)&=\frac{11}{2}.
\end{aligned}
$$

An LP bound belongs to a **subproblem**; it is not necessarily the value of a feasible integer solution.

In [ ]:
def close_polygon(points):
    return np.vstack([points, points[0]])


left_A = np.vstack([A, [1.0, 0.0]])
left_b = np.append(b, 2.0)
right_A = np.vstack([A, [-1.0, 0.0]])
right_b = np.append(b, -3.0)

child_data = [
    (left_A, left_b, left_x, "$Q_L$: $x_1\leq 2$"),
    (right_A, right_b, right_x, "$Q_R$: $x_1\geq 3$"),
]
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
for ax, (child_A, child_b, optimum, title) in zip(axes, child_data, strict=True):
    child_vertices = close_polygon(hrep_to_vertices_2d(child_A, child_b))
    ax.fill(
        closed_vertices[:, 0],
        closed_vertices[:, 1],
        color="tab:blue",
        alpha=0.06,
        label="parent relaxation",
    )
    ax.plot(closed_vertices[:, 0], closed_vertices[:, 1], color="tab:blue", alpha=0.3)
    ax.fill(
        child_vertices[:, 0],
        child_vertices[:, 1],
        color="tab:orange",
        alpha=0.25,
        label="child relaxation",
    )
    ax.plot(child_vertices[:, 0], child_vertices[:, 1], color="tab:orange")
    ax.scatter(lattice_points[:, 0], lattice_points[:, 1], color="black", s=20, zorder=3)
    ax.scatter(*optimum, marker="x", s=100, linewidths=2, color="tab:red", zorder=4)
    ax.set_xlim(-0.5, 5.0)
    ax.set_ylim(-0.5, 5.0)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("$x_1$")
    ax.set_title(title)
    ax.grid(True)
axes[0].set_ylabel("$x_2$")
axes[0].legend(loc="upper left")
fig.suptitle("The branching disjunction partitions the integer points")
fig.tight_layout()
plt.show()

## Fathoming a node

A node is **fathomed** (finished) as soon as one of three conditions applies:

1. **Infeasibility:** its LP relaxation is infeasible, so it contains no integer-feasible point.
2. **Integrality:** its LP optimum satisfies every required integrality condition. It is then optimal for that node and may become the new incumbent.
3. **Bound:** its LP upper bound is no better than the incumbent,
   $$
   z_{\mathrm{LP}}(Q)\le z_{\mathrm{inc}}.
   $$
   No descendant can improve the incumbent, so the entire subtree can be discarded.

Only a feasible, nonintegral node whose bound can still improve the incumbent needs another branch.

## Finding an incumbent on the left branch

We use **depth-first search** and process the left child first. Its LP optimum has $x_2^L=13/4$, so we branch on $x_2$:

$$
Q_{LL}=Q_L\cap\{x_2\le3\},
\qquad
Q_{LR}=Q_L\cap\{x_2\ge4\}.
$$

The first grandchild has the integer LP optimum $(2,3)$ with value $5$. This is our first incumbent. The sibling has LP bound $9/2$, so it cannot improve the incumbent and is pruned by bound.

In [ ]:
left_left_x, left_left_bound = solve_relaxation([(None, 2.0), (None, 3.0)])
left_right_x, left_right_bound = solve_relaxation([(None, 2.0), (4.0, None)])

print(f"x1 <= 2, x2 <= 3: x = {left_left_x}, value = {left_left_bound:.2f}")
print(f"x1 <= 2, x2 >= 4: x = {left_right_x}, bound = {left_right_bound:.2f}")
print("Incumbent value:", left_left_bound)
print("Prune the second node by bound:", left_right_bound <= left_left_bound)

## The complete algorithm

A basic branch-and-bound method for maximization is:

1. Put the root relaxation on a list of pending nodes and start without an incumbent.
2. Select and remove one pending node.
3. Solve its LP relaxation.
4. Fathom it if the LP is infeasible or its bound cannot beat the incumbent.
5. If its LP solution is integer-feasible, update the incumbent and fathom the node.
6. Otherwise, select a fractional integer-constrained variable, create its two children, and add them to the pending list.
7. Repeat until no pending node remains.

The pending-node rule determines the search strategy. A stack gives **depth-first search**, while selecting the largest LP bound gives **best-bound search**. These choices can change the amount of work dramatically, but they do not change the pruning logic.

## Implementation

The small implementation below is kept in `src/optimization/branch_and_bound.py` because later examples can reuse the solver and, especially, its node trace. The example-specific LP calls, plots, and tree rendering remain in this notebook.

The solver handles small pure and mixed-integer maximization problems in inequality form. It uses depth-first search and the first-fractional-variable rule so that the control flow stays close to the algorithm above.

```{literalinclude} ../src/optimization/branch_and_bound.py
:language: python
:linenos:
:start-after: "# --- BRANCH_AND_BOUND:solve:start"
:end-before: "# --- BRANCH_AND_BOUND:solve:end"
```

In [ ]:
from src.optimization.branch_and_bound import branch_and_bound

result = branch_and_bound(c, A, b, bounds=free_bounds)

print("Status:", result.status)
print("Optimal integer solution:", result.x)
print("Optimal objective value:", result.objective)
print("Processed nodes:", len(result.nodes))

The solver returns one optimal point. Here it finds $(2,3)$; the symmetric point $(3,2)$ is also optimal. Branch-and-bound is required to certify the **value** $5$, but it need not enumerate every solution attaining that value.

In [ ]:
def format_point(point):
    if point is None:
        return "—"
    return np.array2string(point, precision=2, suppress_small=True)


trace = pd.DataFrame(
    [
        {
            "node": node.node_id,
            "parent": "—" if node.parent_id is None else node.parent_id,
            "depth": node.depth,
            "LP solution": format_point(node.lp_solution),
            "LP bound": "—" if node.lp_bound is None else f"{node.lp_bound:.2f}",
            "action": node.action,
            "branch variable": (
                "—" if node.branch_variable is None else f"x{node.branch_variable + 1}"
            ),
            "incumbent": (
                "—" if node.incumbent_value is None else f"{node.incumbent_value:.2f}"
            ),
        }
        for node in result.nodes
    ]
)
trace

The table is a certificate of the search logic:

- Node 3 supplies the incumbent value $5$.
- Nodes 4, 7, and 8 are discarded because their LP bounds are at most $5$.
- Node 6 is infeasible.
- Every other processed node is either branched or integer-feasible.

Notice that node numbers encode **creation order**, while the rows encode **processing order**. Depth-first search creates both children but completely explores the left child before returning to its sibling.

In [ ]:
nodes_by_id = {node.node_id: node for node in result.nodes}
children = {node_id: [] for node_id in nodes_by_id}
for node in result.nodes:
    if node.parent_id is not None:
        children[node.parent_id].append(node.node_id)

leaf_counter = iter(range(sum(not child_ids for child_ids in children.values())))
x_positions = {}


def assign_x_position(node_id):
    if not children[node_id]:
        x_positions[node_id] = next(leaf_counter)
    else:
        child_positions = [assign_x_position(child_id) for child_id in children[node_id]]
        x_positions[node_id] = sum(child_positions) / len(child_positions)
    return x_positions[node_id]


assign_x_position(0)
colors = {
    "branched": "tab:blue",
    "integer": "tab:green",
    "bound": "tab:orange",
    "infeasible": "tab:red",
}

fig, ax = plt.subplots(figsize=(12, 6))
for node in result.nodes:
    x = x_positions[node.node_id]
    y = -node.depth
    if node.parent_id is not None:
        parent = nodes_by_id[node.parent_id]
        ax.plot(
            [x_positions[parent.node_id], x],
            [-parent.depth, y],
            color="0.55",
            zorder=1,
        )
    bound_label = "infeasible" if node.lp_bound is None else f"$z_{{LP}}={node.lp_bound:.2f}$"
    branch_label = (
        "" if node.branch_variable is None else f"\nbranch on $x_{node.branch_variable + 1}$"
    )
    label = f"N{node.node_id}\n{bound_label}\n{node.action}{branch_label}"
    ax.text(
        x,
        y,
        label,
        ha="center",
        va="center",
        fontsize=9,
        bbox={
            "boxstyle": "round,pad=0.35",
            "facecolor": colors[node.action],
            "alpha": 0.18,
            "edgecolor": colors[node.action],
        },
        zorder=2,
    )

ax.set_xlim(-0.7, max(x_positions.values()) + 0.7)
ax.set_ylim(-max(node.depth for node in result.nodes) - 0.6, 0.6)
ax.set_title("Complete depth-first branch-and-bound tree")
ax.axis("off")
plt.show()

## Why the final incumbent is optimal

The certificate rests on three simple facts:

1. **Branching is exhaustive for integer points.** Each child disjunction preserves all integer-feasible points of its parent.
2. **LP bounds are valid.** A node's integer-feasible set is contained in its LP relaxation, so no descendant can exceed the node's LP upper bound.
3. **Fathoming is safe.** Infeasible nodes contain no candidates, integral LP optima solve their nodes, and bound-pruned nodes cannot improve the incumbent.

When the pending list becomes empty, every integer-feasible point lies in a fathomed node. None can have value greater than the incumbent, and the incumbent itself is feasible. Therefore its value is optimal.

For a bounded rational integer program, repeated branching eventually isolates or excludes the finitely many integer candidates. Practical solvers additionally rely on stronger relaxations and many implementation enhancements to avoid building an enormous tree.

## Mixed-integer variables

For a mixed-integer problem, only indices in $I$ are tested for integrality and eligible for branching. Continuous variables may remain fractional. In the helper, this set is supplied with the zero-based `integer_indices` argument. For example,

```python
result = branch_and_bound(c, A, b, integer_indices=[0, 2])
```

requires $x_1$ and $x_3$ to be integral while leaving all other variables continuous. Omitting the argument makes every variable integer-constrained.

## What practical solvers add

This implementation deliberately exposes the core idea. Production mixed-integer solvers also use:

- presolve and bound propagation;
- primal heuristics to find incumbents early;
- strong branching and pseudocosts for variable selection;
- best-bound or hybrid node-selection strategies;
- cutting planes, producing **branch-and-cut**;
- warm starts and reusable LP bases;
- numerical safeguards, parallel search, and resource limits.

These additions matter because a branch-and-bound tree can be exponentially large. The implementation here is appropriate for small reproducible examples, not large-scale optimization.

## Takeaways

- An LP relaxation gives an upper bound for a maximization integer program.
- A feasible integer incumbent gives a lower bound.
- Branching on $x_j^*$ uses the valid disjunction $x_j\le\lfloor x_j^*\rfloor$ or $x_j\ge\lceil x_j^*\rceil$.
- A node is fathomed by infeasibility, integrality, or a bound that cannot improve the incumbent.
- When no nodes remain, the incumbent and the exhausted bounds certify global optimality.
- Search and branching rules affect efficiency, while the validity of the bounds and disjunctions ensures correctness.
- Practical solvers strengthen this basic framework with cuts, heuristics, presolve, and advanced search rules.

## Next

Branch-and-bound is unnecessary when the LP relaxation already has an integral optimum. The next notebook studies structural conditions—especially total unimodularity and total dual integrality—that explain when this happens.

➡️ [07 - Unimodularity and Total Dual Integrality](07-Unimodularity-and-Total-Dual-Integrality.ipynb)